# Kandinsky V2A — Multi-Video Listening Study

This notebook generates **two seed candidates for each of 2–5 short video scenes** with the same Kandinsky checkpoint. It then creates a blinded, offline review package.

**What this pilot answers:** for a small set of game clips, do the generated sounds fit the visible event, align in time, sound clean, and seem usable? Does changing the random seed help listeners choose a better candidate?

**This is not model training and not a comparison against ElevenLabs or Stable Audio.** Kandinsky still receives a short event prompt as well as the video. The prompt must describe the intended sound only. We keep the scene and prompt fixed while changing seeds.

## Run order
1. Select an A100 GPU runtime and run the environment setup.
2. Upload 2–5 videos, each at least five seconds long.
3. Set one concise sound-event prompt and optional start time for each uploaded filename.
4. Generate with seeds 42 and 123. Each scene uses the same Lite Distill checkpoint and five-second window.
5. Download the review artifacts ZIP. Share only `Participant_Package.zip`; keep `researcher_key.json` private.

Start with clips where a clear sound event happens within a chosen five-second window. Avoid private/customer footage unless the raters are allowed to view it.

## 1 — Runtime and GPU check

Use the A100 runtime that worked for the existing notebook. This cell stops early if Colab has no GPU or is running an unsupported Python version.

In [ ]:
import os, sys, shutil, subprocess, json, time, hashlib, zipfile
from pathlib import Path
from datetime import datetime, timezone

print("Python:", sys.version)
if sys.version_info < (3, 13):
    raise RuntimeError("This Kandinsky setup expects Python 3.13 or newer. Select the compatible Colab runtime.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("GPU runtime is missing. In Colab, choose Runtime > Change runtime type > A100 GPU.")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode:
    raise RuntimeError("Could not inspect the GPU: " + gpu.stderr[-1000:])
print("GPU:", gpu.stdout.strip())
if "A100" not in gpu.stdout:
    print("This notebook was prepared for the A100 profile that worked in the existing Kandinsky test.")

WORK = Path("/content/kandinsky_multivideo_review")
INPUT_DIR = WORK / "inputs"
WINDOW_DIR = WORK / "scene_windows"
GEN_DIR = WORK / "generated"
BUNDLE = WORK / "kandinsky_review_bundle.zip"
for folder in (INPUT_DIR, WINDOW_DIR, GEN_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Working folder:", WORK)

## 2 — Install the official Kandinsky runner

The notebook uses the same official Kandinsky repository and Lite Distill path as the working single-video notebook. It does not ask for API keys. On a fresh Colab runtime the repository, environment, and model weights must be installed/downloaded once.

In [ ]:
REPO = Path("/content/kandinsky-6")
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/kandinskylab/kandinsky-6.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "uv"], check=True)
if not Path("/usr/local/bin/just").exists():
    subprocess.run(["bash", "-lc", "curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin"], check=True)
subprocess.run(["just", "setup"], cwd=REPO, check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Kandinsky source commit:", SOURCE_COMMIT)
subprocess.run(["just", "download", "lite-distill"], cwd=REPO, check=True)
WEIGHT_REPO_CACHE = Path.home() / ".cache/kandinsky/weights/models--kandinskylab--Kandinsky-6.0-Lite-distill-5s-Diffusers"
CHECKPOINT_REF = WEIGHT_REPO_CACHE / "refs/main"
CHECKPOINT_REVISION = CHECKPOINT_REF.read_text(encoding="utf-8").strip() if CHECKPOINT_REF.is_file() else "not_detected"
print("Lite Distill checkpoint revision:", CHECKPOINT_REVISION)
print("Lite Distill checkpoint is ready.")

## 3 — Upload several source videos

Select 2–5 MP4 files in the upload dialog. Each scene will be trimmed to one five-second window. The source audio is discarded for this pilot; only the video frames condition Kandinsky.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not 2 <= len(uploaded) <= 5:
    raise ValueError(f"Upload 2–5 videos for this pilot; received {len(uploaded)}.")
UPLOADED_VIDEOS = {}
for i, (name, payload) in enumerate(uploaded.items(), start=1):
    if Path(name).suffix.lower() not in {".mp4", ".mov", ".m4v"}:
        raise ValueError(f"Unsupported video type: {name}. Use MP4, MOV, or M4V.")
    stored = INPUT_DIR / f"upload_{i:02d}{Path(name).suffix.lower()}"
    stored.write_bytes(payload)
    UPLOADED_VIDEOS[name] = stored
    print(f"{name} → {stored} ({len(payload) / 1e6:.1f} MB)")
print("Now copy the exact uploaded filenames into the next cell.")

## 4 — Describe the intended sound for each scene

Kandinsky currently needs a text prompt in addition to the video. Keep each prompt short and sound-focused; do not describe camera movement, lighting, or story atmosphere. `start_seconds` selects the five-second segment that contains the event.

Replace the example keys with the exact filenames printed above. The prompt stays fixed across both seeds for that video.

In [ ]:
SCENE_CONFIG = {
    # "wood_block.mp4": {"prompt": "One wooden block hits a wooden floor with a short hollow thud. No music or speech.", "start_seconds": 0},
    # "footsteps.mp4": {"prompt": "Several footsteps on a hard stone floor. No music or speech.", "start_seconds": 4},
}

SEEDS = [42, 123]                 # Same two seed settings for every scene.
GENERATION_SECONDS = 5            # Keep fixed: this is the validated short-window setting.
CHECKPOINT = "lite-distill"

if set(SCENE_CONFIG) != set(UPLOADED_VIDEOS):
    missing = sorted(set(UPLOADED_VIDEOS) - set(SCENE_CONFIG))
    extra = sorted(set(SCENE_CONFIG) - set(UPLOADED_VIDEOS))
    raise ValueError(f"Add exactly one SCENE_CONFIG entry per upload. Missing={missing}; unknown={extra}")
if len(SEEDS) != 2 or len(set(SEEDS)) != 2:
    raise ValueError("Keep exactly two distinct seeds for this first listener pilot.")
for name, config in SCENE_CONFIG.items():
    if not isinstance(config.get("prompt"), str) or not config["prompt"].strip():
        raise ValueError(f"A short sound-event prompt is required for {name}.")
    if float(config.get("start_seconds", 0)) < 0:
        raise ValueError(f"start_seconds cannot be negative for {name}.")
print(f"Plan: {len(SCENE_CONFIG)} scenes × {len(SEEDS)} seeds = {len(SCENE_CONFIG) * len(SEEDS)} candidates.")

## 5 — Generate the candidates

For each scene, the notebook prepares the selected five-second muted video segment. It holds the checkpoint, prompt, duration, and other settings constant, then changes only the seed. Kandinsky is invoked once per candidate; on a fresh process it may reload weights each time, so expect several sequential runs.

In [ ]:
DEVICE_CONFIG = REPO / "kandinsky/configs/devices/a100-lite-distill.yaml"
DEVICE_CONFIG.write_text("checkpoint: lite-distill\ncache:\n  mode: none\noffload:\n  strategy: module\ncompile:\n  strategy: none\nattention:\n  engine: sdpa\ntext_embedder:\n  quantized_qwen: false\nbeautifier:\n  name: none\n", encoding="utf-8")

def run_command(command, cwd=None):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-2500:])
    if result.returncode:
        if result.stderr:
            print(result.stderr[-5000:])
        raise RuntimeError("A Kandinsky or ffmpeg command failed. Read the error above before retrying.")
    return result

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

scenes, candidates = [], []
candidate_number = 0
for scene_number, (original_name, source_path) in enumerate(UPLOADED_VIDEOS.items(), start=1):
    scene_id = f"scene_{scene_number:02d}"
    scene_config = SCENE_CONFIG[original_name]
    start_seconds = float(scene_config.get("start_seconds", 0))
    probe = run_command(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(source_path)])
    source_duration = float(probe.stdout.strip())
    if start_seconds + GENERATION_SECONDS > source_duration:
        raise ValueError(f"{original_name} is too short for start={start_seconds}s and a five-second window ({source_duration:.2f}s total).")
    scene_video = WINDOW_DIR / f"{scene_id}.mp4"
    run_command(["ffmpeg", "-y", "-ss", str(start_seconds), "-i", str(source_path), "-t", str(GENERATION_SECONDS), "-map", "0:v:0", "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-r", "24", str(scene_video)])
    scenes.append({"scene_id": scene_id, "label": f"Scene {scene_number}", "source_name": original_name,
                   "source_sha256": sha256_file(source_path), "video_path": f"videos/{scene_id}.mp4",
                   "video_sha256": sha256_file(scene_video), "start_seconds": start_seconds,
                   "duration_seconds": GENERATION_SECONDS, "event_prompt": scene_config["prompt"]})
    for seed in SEEDS:
        candidate_number += 1
        candidate_id = f"candidate_{candidate_number:03d}"
        output_mp4 = GEN_DIR / f"{candidate_id}.mp4"
        output_wav = GEN_DIR / f"{candidate_id}.wav"
        command = ["just", "generate", scene_config["prompt"], "--config", str(DEVICE_CONFIG),
                   "--attention-engine", "sdpa", "--video-file", str(scene_video),
                   "--time-length", str(GENERATION_SECONDS), "--out", str(output_mp4), "--seed", str(seed)]
        print(f"\nGenerating {candidate_id}: scene={scene_number}, seed={seed}")
        print("Command:", " ".join(command))
        started = time.time()
        run_command(command, cwd=REPO)
        generation_seconds = round(time.time() - started, 2)
        run_command(["ffmpeg", "-y", "-i", str(output_mp4), "-vn", "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(output_wav)])
        candidates.append({"candidate_id": candidate_id, "scene_id": scene_id, "seed": seed,
                           "candidate_index": len([c for c in candidates if c["scene_id"] == scene_id]) + 1,
                           "event_prompt": scene_config["prompt"], "audio_path": f"audio/{candidate_id}.wav",
                           "audio_sha256": sha256_file(output_wav), "generated_mp4_path": f"generated/{candidate_id}.mp4",
                           "output_mp4_sha256": sha256_file(output_mp4), "generation_seconds": generation_seconds})
        print(f"Finished {candidate_id} in {generation_seconds:.1f} seconds.")

print(f"Generated {len(candidates)} candidates for {len(scenes)} scenes.")

## 6 — Package the blind listener study

The research bundle keeps prompts, seed values, source filenames, and model revisions for analysis. The participant package hides seed and prompt labels, randomizes scene/candidate order per person, and contains only the muted five-second video clips and generated WAVs. It works offline after extraction.

In [ ]:
PRODUCT_REPO = Path("/content/kandinsky-v2a-colab")
if not (PRODUCT_REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/AytugSahinkanat/kandinsky-v2a-colab.git", str(PRODUCT_REPO)], check=True)

manifest = {
    "schema": "kandinsky-v2a-review-bundle-v1",
    "study_title": "Kandinsky Video-to-Audio Seed Pilot",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model": {"name": "Kandinsky 6.0", "checkpoint": CHECKPOINT,
              "checkpoint_repo": "kandinskylab/Kandinsky-6.0-Lite-distill-5s-Diffusers",
              "checkpoint_revision": CHECKPOINT_REVISION},
    "source_repo": "https://github.com/kandinskylab/kandinsky-6",
    "source_commit": SOURCE_COMMIT,
    "generation": {"duration_seconds": GENERATION_SECONDS, "seeds": SEEDS,
                    "prompt_expansion": "disabled", "attention_engine": "sdpa"},
    "scenes": scenes, "candidates": candidates,
}
BUNDLE.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2))
    for scene in scenes:
        archive.write(WINDOW_DIR / f"{scene['scene_id']}.mp4", scene["video_path"])
    for candidate in candidates:
        archive.write(GEN_DIR / f"{candidate['candidate_id']}.wav", candidate["audio_path"])
        archive.write(GEN_DIR / f"{candidate['candidate_id']}.mp4", candidate["generated_mp4_path"])
print("Research bundle:", BUNDLE, f"({BUNDLE.stat().st_size / 1e6:.1f} MB)")

STUDY_DIR = WORK / ("participant_study_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
builder = PRODUCT_REPO / "tools/build_kandinsky_review.py"
if not builder.is_file():
    raise FileNotFoundError(f"Review builder is missing from the public repository: {builder}")
run_command([sys.executable, str(builder), str(BUNDLE), str(STUDY_DIR)])
participant_zip = STUDY_DIR / "Participant_Package.zip"
researcher_key = STUDY_DIR / "researcher_key.json"
print("Participant ZIP:", participant_zip)
print("Private researcher key:", researcher_key)
print("IMPORTANT: share only Participant_Package.zip with raters. Keep researcher_key.json and this Colab folder private.")

## 7 — Download the study files

The combined download contains the participant ZIP, the private seed/prompt key, and the full generation bundle. **Do not forward this combined archive to participants.** Extract it and share only the nested `Participant_Package.zip`. Keep `researcher_key.json` for combining returned JSON ratings.

In [ ]:
from google.colab import files
from IPython.display import display, FileLink

DOWNLOAD = WORK / "Kandinsky_Review_Researcher_Artifacts.zip"
with zipfile.ZipFile(DOWNLOAD, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(participant_zip, "Participant_Package.zip")
    archive.write(researcher_key, "researcher_key.json")
    archive.write(BUNDLE, "generation_bundle.zip")
print("Download this archive and extract it. Give participants only Participant_Package.zip.")
files.download(str(DOWNLOAD))

## What to look for

- **Scene fit:** does the sound describe the event shown?
- **Timing:** does the important sound happen at the right moment?
- **Quality:** is it convincing and clean enough to consider using?
- **Noise/artifacts:** are there unwanted textures, distortion, or repeated fragments?
- **Keep decision:** would you use this candidate as-is, after editing, or not at all?

This first review rates Kandinsky seed alternatives only. It does not compare the current Orfeo/ElevenLabs workflow yet. Once these responses show which scenes and candidate types work, we can add the existing workflow as a blinded baseline using the same source clips and rating form. A small number of raters is directional feedback, not a statistically reliable success percentage.